## Generate YOLOv8 dataset

In [ ]:
!pip install pyedflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 7.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
## NO OVERLAP ##

import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy.signal import convolve
import xml.etree.ElementTree as ET
import pyedflib

# Define base path and subjects
BASE_DIR = r"/content/drive/MyDrive/Yu's Project/NCKUSH_50sub/EDF_XML/"
#OUTPUT_DIR = r"/content/drive/MyDrive/Yui's Project/NCKUSH_50sub/Dataset/"

# severe = ['s3','s4','s6','s7','s9','s12','s15','s18','s20','s21','s22','s24','s25','s29','s35','s39','s41','s48']
# moderate = ['s1','s2','s5','s8','s10','s11','s14','s16','s19','s23']
# mind = ['s13','s17','s28','s31','s33','s42','s43','s45','s46','s46','s47','s49','s50']
# no_apnea = ['s13','s17','s28','s31','s33','s42','s43','s45','s46','s46','s47','s49','s50']

subjects = ['s46']

def load_and_process_events(xml_file_path, total_duration):
    tree = ET.parse(xml_file_path)
    root = tree.getroot()
    events = []

    # Load labeled events
    for event in root.findall('.//ScoredEvent'):
        start = float(event.find('Start').text)
        duration = float(event.find('Duration').text)
        end = start + duration
        event_name = event.find('Name').text
        if 'Apnea' in event_name or 'Hypopnea' in event_name:
            events.append((start, end, 1))  # 1 for Apnea/Hypopnea

    # Sort events by start time
    events.sort(key=lambda x: x[0])

    # Fill in the gaps with normal events
    filled_events = []
    last_end = 0
    for start, end, event_class in events:
        if start > last_end:
            filled_events.append((last_end, start, 0))  # 0 for Normal
        filled_events.append((start, end, event_class))
        last_end = end

    # Ensure the timeline is filled to the end
    if last_end < total_duration:
        filled_events.append((last_end, total_duration, 0))  # 0 for Normal

    return filled_events

def MorletWavelet(fc, fs, width=7):
    sigma_t = 1 / (2 * np.pi * (fc / width))
    t = np.arange(-3.5 * sigma_t * fs, 3.5 * sigma_t * fs + 1) / fs
    A = 1 / (sigma_t * np.sqrt(2 * np.pi))
    return A * np.exp(-t**2 / (2 * sigma_t**2)) * np.exp(1j * 2 * np.pi * fc * t)

def tfa_morlet(signal_x, fs, fmin, fmax, fstep):
    frequencies = np.arange(fmin, fmax + fstep, fstep)
    TFmap = np.zeros((len(frequencies), len(signal_x)), dtype=np.complex128)

    for i, fc in enumerate(frequencies):
        wavelet = MorletWavelet(fc, fs)
        conv_result = convolve(signal_x, wavelet, mode='same')
        TFmap[i, :] = conv_result

    return frequencies, np.abs(TFmap)

SEGMENT_DURATION_SEC = 90

# Process each subject
for subject in subjects:
    edf_file_path = os.path.join(BASE_DIR, f'{subject}.edf')
    xml_file_path = os.path.join(BASE_DIR, f'{subject}_re.XML')
    #OUTPUT_DIR = f'donothing_640_1280_{subject}_90s_moderate'
    OUTPUT_DIR = os.path.join(r"/content/drive/MyDrive/Yui's Project/NCKUSH_50sub/Dataset/", f'donothing_640_1280_{subject}_90s_no_apnea')

    # Ensure the output directory exists
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    # Load ECG data
    with pyedflib.EdfReader(edf_file_path) as f:
        input_channel = 9  # Adjust based on your EDF file structure
        samp_freq = f.getSampleFrequency(input_channel)
        ecg_signal = f.readSignal(input_channel)

    # Filter the ECG signal
    ecg_filtered = signal.filtfilt(*signal.butter(4, 0.5 / (samp_freq / 2), btype='high'), ecg_signal)

    # Apply Z-score normalization (mean=0, std=1)
    ecg_filtered = (ecg_filtered - np.mean(ecg_filtered)) / np.std(ecg_filtered)

    # Total duration of the ECG recording in seconds
    total_duration_sec = len(ecg_filtered) / samp_freq

    # Load events and fill gaps with normal
    events = load_and_process_events(xml_file_path, total_duration_sec)

    num_segments = int(np.ceil(total_duration_sec / SEGMENT_DURATION_SEC))

    for i in range(num_segments):
        segment_start = i * SEGMENT_DURATION_SEC
        segment_end = min((i + 1) * SEGMENT_DURATION_SEC, total_duration_sec)
        segment_ecg = ecg_filtered[int(segment_start * samp_freq):int(segment_end * samp_freq)]
        if len(segment_ecg) < SEGMENT_DURATION_SEC * samp_freq:
            break


        frequencies, spectrogram = tfa_morlet(segment_ecg, samp_freq, 0.8, 10, 0.01)

        # ----- Build subject + absolute-time filename base -----
        start_sec = int(segment_start)
        end_sec = int(segment_end)
        base_name = f"{subject}_segment_{start_sec:05d}_to_{end_sec:05d}_spectrogram"

        # Spectrogram image generation and saving
        plt.figure(figsize=(16.8, 8.4))  # ~1280x640 px before tight-crop at dpi=100
        plt.imshow(spectrogram, aspect='auto',
                   extent=[0, SEGMENT_DURATION_SEC, frequencies[0], frequencies[-1]],
                   cmap='inferno', origin='lower')
        plt.axis('off')
        image_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
        plt.savefig(image_path, bbox_inches='tight', pad_inches=0, dpi=100)
        plt.close()

        # Preparing annotations for the current segment
        annotations = []
        for event_start, event_end, event_class in events:
            if event_start < segment_end and event_end > segment_start:
                adjusted_start = max(event_start, segment_start) - segment_start
                adjusted_end   = min(event_end, segment_end)   - segment_start
                normalized_start = adjusted_start / SEGMENT_DURATION_SEC
                normalized_end   = adjusted_end   / SEGMENT_DURATION_SEC
                x_center = (normalized_start + normalized_end) / 2
                width    = normalized_end - normalized_start
                annotations.append(f"{event_class} {x_center:.6f} 0.5 {width:.6f} 1.0\n")

        # Saving annotations with the same base name
        annotation_path = os.path.join(OUTPUT_DIR, f"{base_name}.txt")
        with open(annotation_path, 'w') as file:
            file.writelines(annotations)

    print(f"Processed subject {subject} with segments and annotations.")

Processed subject s46 with segments and annotations.


## ARRANGE files for SUBJECT_WISED 5_FOLD CV and create YAML

In [ ]:
import os
import shutil
import yaml

# Define dataset paths
source_root = r"C:\Users\WTMH\A_YuFlie\donothing_640_1280__s{subject}_90s_normal"
dest_root = r"D:\5fold_ncku_re_yolov8_45s_640_640"

# Define subject splits for each fold
folds = {
    1: {"valid": list(range(1, 11)), "train": list(range(11, 51))},
    2: {"valid": list(range(11, 21)), "train": list(range(1, 11)) + list(range(21, 51))},
    3: {"valid": list(range(21, 31)), "train": list(range(1, 21)) + list(range(31, 51))},
    4: {"valid": list(range(31, 41)), "train": list(range(1, 31)) + list(range(41, 51))},
    5: {"valid": list(range(41, 51)), "train": list(range(1, 41))}
}

# Function to copy files
def copy_files(subjects, dest_path):
    os.makedirs(os.path.join(dest_path, "images"), exist_ok=True)
    os.makedirs(os.path.join(dest_path, "labels"), exist_ok=True)

    for subj in subjects:
        source_path = source_root.format(subject=subj)
        if not os.path.exists(source_path):
            print(f"Skipping missing subject: s{subj}")
            continue

        for file in os.listdir(source_path):
            src_file = os.path.join(source_path, file)
            if file.endswith(".png"):
                shutil.copy(src_file, os.path.join(dest_path, "images", file))
            elif file.endswith(".txt"):
                shutil.copy(src_file, os.path.join(dest_path, "labels", file))

# Function to create YAML file
def create_yaml_file(fold_path, fold_name):
    yaml_data = {
        "path": f"{fold_name}/",
        "train": "../train/images",
        "val": "../valid/images",
        "test": "../test/images",
        "names": {
            0: "N",
            1: "AH"
        }
    }

    yaml_path = os.path.join(fold_path, f"{fold_name}.yaml")
    with open(yaml_path, 'w') as yaml_file:
        yaml.dump(yaml_data, yaml_file, default_flow_style=False)

# Create datasets for each fold
for fold, data in folds.items():
    fold_name = f"re_kfold_fold{fold}_45s"
    fold_path = os.path.join(dest_root, fold_name)

    # Copy training data
    train_path = os.path.join(fold_path, "train")
    copy_files(data["train"], train_path)

    # Copy validation data
    valid_path = os.path.join(fold_path, "valid")
    copy_files(data["valid"], valid_path)

    # Create YAML file
    create_yaml_file(fold_path, fold_name)

    print(f"Finished processing fold {fold} - YAML created at {fold_path}")


## ARRANGE files for LOSO CV and create YAML

In [ ]:
import os
import shutil
import yaml

# Define dataset paths
source_root = r"/content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat"
dest_root = r"/content/drive/MyDrive/Yu's Project/loso1to5"

# Total number of subjects
total_subjects = 50

# === 👇 CONFIGURE THIS RANGE 👇 ===
start_fold = 1   # Start from this subject (inclusive)
end_fold = 5     # End at this subject (inclusive)
# ==================================

# Function to copy files
def copy_files(subjects, dest_path):
    os.makedirs(os.path.join(dest_path, "images"), exist_ok=True)
    os.makedirs(os.path.join(dest_path, "labels"), exist_ok=True)

    for subj in subjects:
        source_path = source_root.format(subject=subj)
        if not os.path.exists(source_path):
            print(f"Skipping missing subject: s{subj}")
            continue

        for file in os.listdir(source_path):
            src_file = os.path.join(source_path, file)
            if file.endswith(".png"):
                shutil.copy(src_file, os.path.join(dest_path, "images", file))
            elif file.endswith(".txt"):
                shutil.copy(src_file, os.path.join(dest_path, "labels", file))

# Function to create YAML file with proper paths
def create_yaml_file(yaml_output_path, fold_name):
    yaml_data = {
        "train": f"/home/datasets/{fold_name}/train/images",
        "val": f"/home/datasets/{fold_name}/valid/images",
        "test": f"/home/datasets/{fold_name}/test/images",  # Optional: can be removed if not used
        "names": {
            0: "N",
            1: "AH"
        }
    }

    with open(yaml_output_path, 'w') as yaml_file:
        yaml.dump(yaml_data, yaml_file, default_flow_style=False)

# Generate LOSO folds within selected range
for subject in range(start_fold, end_fold + 1):
    fold_dirname = f"re_loso_fold_s{subject:02d}_90s"
    fold_path = os.path.join(dest_root, fold_dirname)
    yaml_path = os.path.join(dest_root, f"{fold_dirname}.yaml")  # YAML outside the fold folder

    # Subjects
    train_subjects = [s for s in range(1, total_subjects + 1) if s != subject]
    valid_subjects = [subject]

    # Create folders and copy data
    train_path = os.path.join(fold_path, "train")
    valid_path = os.path.join(fold_path, "valid")
    test_path = os.path.join(fold_path, "test")  # Optional placeholder

    copy_files(train_subjects, train_path)
    copy_files(valid_subjects, valid_path)
    os.makedirs(os.path.join(test_path, "images"), exist_ok=True)  # Ensure test/images exists

    # Create YAML config outside the fold folder
    create_yaml_file(yaml_path, fold_dirname)

    print(f"✅ Fold s{subject:02d} created: {fold_path}")
    print(f"📄 YAML created: {yaml_path}")


✅ Fold s01 created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s01_90s
📄 YAML created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s01_90s.yaml
✅ Fold s02 created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s02_90s
📄 YAML created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s02_90s.yaml
✅ Fold s03 created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s03_90s
📄 YAML created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s03_90s.yaml
✅ Fold s04 created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s04_90s
📄 YAML created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s04_90s.yaml
✅ Fold s05 created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s05_90s
📄 YAML created: /content/drive/MyDrive/Yu's Project/loso1to5/re_loso_fold_s05_90s.yaml


Getout Folder

In [ ]:
import os
import shutil

# Path dataset awal
base_path = "/content/drive/MyDrive/Yu's Project/Spectogram/0.8-10"
# Path dataset baru yang datar
dest_path = "/content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat"

os.makedirs(dest_path, exist_ok=True)

categories = ['mild', 'moderate', 'other', 'severe']

for cat in categories:
    cat_path = os.path.join(base_path, cat)
    for subj in os.listdir(cat_path):
        subj_path = os.path.join(cat_path, subj)
        if os.path.isdir(subj_path):
            # Buat nama folder baru: sX_category
            new_subj_name = f"{subj}_{cat}"
            new_subj_path = os.path.join(dest_path, new_subj_name)
            shutil.move(subj_path, new_subj_path)
            print(f"Moved {subj_path} -> {new_subj_path}")


Moved /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10/mild/s13 -> /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat/s13_mild
Moved /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10/mild/s17 -> /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat/s17_mild
Moved /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10/mild/s31 -> /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat/s31_mild
Moved /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10/mild/s33 -> /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat/s33_mild
Moved /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10/mild/s43 -> /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat/s43_mild
Moved /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10/mild/s45 -> /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat/s45_mild
Moved /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10/mild/s47 -> /content/drive/MyDrive/Yu's Project/Spectogram/0.8-10_flat/s47_mild
Moved /content/drive